In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [14]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.6 MB/s eta 0:00:00


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_PATH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

print("Project path:", PROJECT_PATH)

Project path: /content


In [19]:
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/rag-chatbot-evaluation-framework")
SRC_PATH = PROJECT_DIR / "src"

print("PROJECT_PATH exists:", PROJECT_DIR.exists())
print("SRC_PATH exists:", SRC_PATH.exists())
print("Files in src:", os.listdir(SRC_PATH))

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from document_loader import load_markdown_documents, validate_documents
from text_splitter import create_chunks
from vector_store import FaissVectorStore
from rag_pipeline import SimpleRAGPipeline
from evaluator import add_pass_fail_flags, evaluate_dataset, identify_failed_cases, save_evaluation_outputs, summarize_results

DOCUMENT_DIR = PROJECT_DIR / "data" / "documents"
EVALUATION_FILE = PROJECT_DIR / "data" / "evaluation" / "evaluation_questions.csv"
RESULTS_DIR = PROJECT_DIR / "results"

docs = load_markdown_documents(str(DOCUMENT_DIR))
validate_documents(docs)
chunks = create_chunks(docs)

vector_store = FaissVectorStore()
vector_store.build_index(chunks)
rag = SimpleRAGPipeline(vector_store, top_k=3)

evaluation_df = pd.read_csv(EVALUATION_FILE)
print('Evaluation questions:', len(evaluation_df))
evaluation_df.head()

PROJECT_PATH exists: True
SRC_PATH exists: True
Files in src: ['__init__.py', 'rag_pipeline.py', 'text_splitter.py', 'document_loader.py', 'vector_store.py', '__pycache__', 'evaluator.py', 'report_generator.py']
Document validation passed. Loaded 5 documents.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evaluation questions: 32


,question,expected_answer,expected_source,expected_keywords,question_type
0,What is the return period?,Customers can return most products within 30 d...,return_policy.md,30 days;return;delivery,normal
1,What condition must returned items be in?,"Returned items must be unused, undamaged, and ...",return_policy.md,unused;undamaged;original packaging,normal
2,How long does it usually take to process a ref...,Refunds are usually processed within 5 to 10 b...,return_policy.md,5 to 10 business days;refund;received,normal
3,Can customized products be returned?,Customized products cannot be returned.,return_policy.md,customized products;cannot be returned,normal
4,What should a customer do if a product arrives...,Customers should contact customer support with...,return_policy.md,7 days;customer support;photos,normal


In [21]:
evaluation_results = evaluate_dataset(rag, evaluation_df)
evaluation_results = add_pass_fail_flags(evaluation_results)
evaluation_results.head()

,question,question_type,answer,expected_answer,expected_source,retrieved_sources,source_match,keyword_recall,matched_keywords,missing_keywords,unanswerable_safe,source_pass,keyword_pass,unanswerable_pass,overall_pass
0,What is the return period?,normal,# Return Policy\n\nCustomers can return most p...,Customers can return most products within 30 d...,return_policy.md,return_policy.md;payment_policy.md;warranty_po...,1.0,1.0,30 days;return;delivery,,NaN,1.0,1,NaN,1
1,What condition must returned items be in?,normal,"Returned items must be unused, undamaged, and ...","Returned items must be unused, undamaged, and ...",return_policy.md,return_policy.md;return_policy.md;payment_poli...,1.0,1.0,unused;undamaged;original packaging,,NaN,1.0,1,NaN,1
2,How long does it usually take to process a ref...,normal,Refunds are usually processed within 5 to 10 b...,Refunds are usually processed within 5 to 10 b...,return_policy.md,return_policy.md;return_policy.md;payment_poli...,1.0,1.0,5 to 10 business days;refund;received,,NaN,1.0,1,NaN,1
3,Can customized products be returned?,normal,"Final sale items, gift cards, and customized p...",Customized products cannot be returned.,return_policy.md,return_policy.md;warranty_policy.md;return_pol...,1.0,1.0,customized products;cannot be returned,,NaN,1.0,1,NaN,1
4,What should a customer do if a product arrives...,normal,ld contact customer support within 7 days of d...,Customers should contact customer support with...,return_policy.md,return_policy.md;warranty_policy.md;warranty_p...,1.0,1.0,7 days;customer support;photos,,NaN,1.0,1,NaN,1


In [22]:
summary = summarize_results(evaluation_results)
summary

,total_questions,answerable_questions,unanswerable_questions,avg_source_match,avg_keyword_recall,avg_unanswerable_safe,overall_pass_rate
0,32,29,3,1.0,0.875,0.0,0.875


In [28]:
failed_cases = identify_failed_cases(evaluation_results)
failed_cases[['question', 'question_type', 'failure_type', 'expected_source', 'retrieved_sources', 'keyword_recall', 'missing_keywords']].head(20)


,question,question_type,failure_type,expected_source,retrieved_sources,keyword_recall,missing_keywords
9,Why might shipping be delayed?,normal,Answer failure: expected keywords missing,shipping_policy.md,shipping_policy.md;shipping_policy.md;return_p...,0.0,public holidays;extreme weather;high order volume
29,Can I return a product after 90 days?,unanswerable,Answer failure: expected keywords missing,none,return_policy.md;warranty_policy.md;warranty_p...,0.0,not provide;90 days;cannot confirm
30,Does the company provide international shipping?,unanswerable,Answer failure: expected keywords missing,none,shipping_policy.md;payment_policy.md;shipping_...,0.0,not mention;international shipping
31,Can I pay with cryptocurrency?,unanswerable,Answer failure: expected keywords missing,none,payment_policy.md;payment_policy.md;warranty_p...,0.0,not mention;cryptocurrency;payment method


In [29]:
output_files = save_evaluation_outputs(evaluation_results, output_dir = str(RESULTS_DIR))
output_files

{'evaluation_results': '/content/drive/MyDrive/rag-chatbot-evaluation-framework/results/evaluation_results.csv',
 'failed_cases': '/content/drive/MyDrive/rag-chatbot-evaluation-framework/results/failed_cases.csv',
 'summary_report': '/content/drive/MyDrive/rag-chatbot-evaluation-framework/results/summary_report.csv'}